# 02 · Data quality and cleaning
Profile the raw extract, apply rules, and prove the clean layer is sound. Rule of thumb: **fix only what is certain, quarantine what would be a guess, log everything.**

In [1]:
import sys, pathlib, json
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from src.data_loader import get_engine, query, PROJECT_ROOT
from src import analysis as an
an.set_style()
pd.set_option("display.float_format", "{:,.2f}".format); pd.set_option("display.max_columns", 30)
engine = get_engine()
o = an.sql_output

In [2]:
o('dq_a01_row_counts_and_duplicates')

,table_name,raw_rows,distinct_keys,extra_rows
0,fee_schedule,65,65,0
1,customers,30060,30000,60
2,accounts,38034,38034,0
3,transactions,2028818,2022818,6000


In [3]:
o('dq_a02_label_variants')

,field,raw_label,rows
0,accounts.account_type,savings,147
1,accounts.account_type,current,100
2,accounts.account_type,digital wallet,46
3,accounts.account_type,fixed savings,7
4,customers.segment,MASS RETAIL,160
5,customers.segment,YOUTH,92
6,customers.segment,SALARIED,67
7,customers.segment,PREMIER,27
8,transactions.channel,mobile app,4371
9,transactions.channel,APP,4332


In [4]:
o('dq_a03_missing_values')

,field,missing,rows
0,customers.county,700,30060
1,transactions.processing_ms = -1 (not captured),3015,2028818
2,transactions.failure_reason (failed rows),901,71645
3,transactions.failure_reason present on success...,250,1957173


### Amounts by channel

In [5]:
o('dq_a04_amount_profile_by_channel')

,channel,rows,non_positive,median_kes,p99_9_kes,max_kes,above_40k
0,ATM,223750,1,"3,500.00","40,000.00","4,000,000.00",35
1,Agent,236593,5,"3,846.00","189,006.71","990,000.00",10178
2,Internet Banking,231032,7,"23,652.50","990,000.00","990,000.00",86306
3,Mobile App,861758,19,"1,782.00","575,396.92","990,000.00",45364
4,USSD,319898,5,975.00,"154,777.41","990,000.00",3256
5,Wallet Link,125690,1,"2,628.00","137,730.03","897,088.00",2573


ATM withdrawals are capped at KES 40,000 per transaction, and every ATM amount above that is exactly 100× a valid amount: a keying error. Those rows, the non-positive amounts (whose sign can't be trusted) and orphan account IDs are **quarantined** in `bank.quarantined_transactions` rather than repaired.

In [6]:
o('dq_a05_timestamp_range')

,year,rows,first_ts,last_ts
0,2025,750705,2025-05-01 00:00:39,2025-12-31 23:59:33
1,2026,1278088,2026-01-01 00:01:19,2026-10-31 23:58:59
2,2035,12,2035-05-23 16:38:01,2035-12-16 10:02:35
3,2036,13,2036-02-25 16:20:19,2036-10-27 10:53:38


In [7]:
o('dq_a06_onboarding_dates')

,future_dates,earliest,latest
0,15,2021-05-03,2066-06-19


### Rules applied (`src/data_cleaning.py`)
| Problem | Rule | Why |
|---|---|---|
| Exact duplicate rows | keep one | identical in every field |
| Channel, status, segment, account-type label variants | map to the reference list; the build **fails** on an unknown label | no silent new categories |
| Onboarding year 2065 etc. | replace with the customer's first account open date | a trusted field exists |
| Duplicate customer IDs whose copies differ | keep the most complete copy | one copy had a blank county |
| Failed attempt without a reason | `Unknown` | keeps the row, owner unknown |
| Successful attempt with a reason | reason → NULL | a success has no failure reason |
| Processing time −1 | NULL | not captured, not zero |
| Orphan account, timestamp after the extract, non-positive amount, ATM amount > 40,000 | **quarantine** | fixing would be a guess |
| **Fee charged against the tariff** | **kept as recorded** | it is a business finding (possible refunds), analysed in notebook 05 |

In [8]:
o('dq_b03_exception_log_summary')

,source_table,issue,action_taken,rows
0,accounts,account type label variant,standardised,300
1,customers,missing county,left NULL (reported as Unknown),696
2,customers,segment label variant,standardised,346
3,customers,exact duplicate row,removed,56
4,customers,onboarding date outside data range (year typo),replaced with first account open date,15
5,customers,duplicate customer_id with differing values,kept the most complete copy,4
6,transactions,channel label variant,standardised,30000
7,transactions,status label variant,standardised,8000
8,transactions,exact duplicate row,removed,6000
9,transactions,processing time sentinel (-1),set to NULL,3000


### Did cleaning catch every injected defect?
The generator recorded how many defects it injected. The exception log should account for each one.

In [9]:
man = json.loads((PROJECT_ROOT / "data/synthetic/raw/injection_manifest.json").read_text())["injected"]
log = o('dq_b03_exception_log_summary').groupby('issue').rows.sum()
pairs = [("transactions_duplicate_rows", "exact duplicate row", "transactions"),
         ("transactions_channel_label_variants", "channel label variant", None),
         ("transactions_status_variants", "status label variant", None),
         ("transactions_failed_missing_reason", "failed transaction without a failure reason", None),
         ("transactions_success_with_reason", "successful transaction carrying a failure reason", None),
         ("transactions_negative_amount", "non-positive amount (sign cannot be trusted)", None),
         ("transactions_atm_amount_x100", "ATM amount above KES 40,000 limit (x100 pattern)", None),
         ("transactions_future_timestamp", "timestamp after extract date (year typo)", None),
         ("transactions_orphan_account", "account_id not in account master", None),
         ("transactions_latency_sentinel", "processing time sentinel (-1)", None),
         ("customers_onboarding_year_typo", "onboarding date outside data range (year typo)", None),
         ("customers_missing_county", "missing county", None),
         ("accounts_type_case", "account type label variant", None)]
ex = o('dq_b03_exception_log_summary')
rows = []
for m, issue, table in pairs:
    sel = ex[ex.issue == issue]
    if table: sel = sel[sel.source_table == table]
    rows.append((m, man[m], int(sel.rows.sum())))
chk = pd.DataFrame(rows, columns=["injected_defect", "injected", "logged"]); chk["difference"] = chk.logged - chk.injected; chk

,injected_defect,injected,logged,difference
0,transactions_duplicate_rows,6000,6000,0
1,transactions_channel_label_variants,30000,30000,0
2,transactions_status_variants,8000,8000,0
3,transactions_failed_missing_reason,900,900,0
4,transactions_success_with_reason,250,250,0
5,transactions_negative_amount,40,40,0
6,transactions_atm_amount_x100,35,35,0
7,transactions_future_timestamp,25,25,0
8,transactions_orphan_account,30,30,0
9,transactions_latency_sentinel,3000,3000,0


Every transaction defect is accounted for exactly. Missing counties are slightly lower than injected because a few blank counties fell on duplicate customer copies, and the more complete copy was kept. (Segment variants are not in this table: upper-casing `SME` changes nothing, so fewer than the 400 injected labels actually differ.)

In [10]:
o('dq_b01_integrity_summary')

,check_name,failures
0,transaction before its account was opened,0
1,transaction before the customer was onboarded,0
2,account opened before customer onboarding,0
3,customer without exactly one primary account,0
4,transaction outside the extract window,0
5,"ATM amount above KES 40,000",0
6,transaction type not offered on its channel (n...,0
7,transaction matching more than one tariff row,0


In [11]:
o('dq_b02_reconciliation')

,transactions,rows_in_semantic_view,value_in_transactions,value_in_customer_month,fees_in_transactions,fees_in_customer_month,attempts_in_channel_hour
0,2022688,2022688,"29,326,528,004.00","29,326,528,004.00","51,717,546.00","51,717,546.00","2,022,688.00"


All eight integrity checks return 0, and the semantic layer carries every transaction once: value and fees in the customer-month grid equal the transaction table to the shilling.